# Intelligent Email Spam Detection Using Machine Learning
### Sreyas Institute of Engineering & Technology — Dept. of CSE (AI & ML)
**Student:** GANTA BALA AMOGH RAJ  
This notebook demonstrates the end-to-end training, evaluation, and serialization of our Email Spam Detection system.

In [ ]:
import pandas as pd
import numpy as np
import re, string, nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score

nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully!")

In [ ]:
# 1. Download & Load Benchmark Dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
print("Dataset Shape:", df.shape)
print(df['label'].value_counts())

In [ ]:
# 2. Text Preprocessing
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
stop_words.discard('free')
stop_words.discard('win')
stop_words.discard('urgent')

def clean_email(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' httpaddr ', text)
    text = re.sub(r'[\$£€¥]', ' currval ', text)
    text = re.sub(r'\b\d+\b', ' numtoken ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [stemmer.stem(w) for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

df['clean_message'] = df['message'].apply(clean_email)
print("Sample cleaned:", df['clean_message'].iloc[0])

In [ ]:
# 3. Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_message'], df['label_num'], test_size=0.20, random_state=42, stratify=df['label_num']
)

# 4. TF-IDF Vectorization
vec = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), sublinear_tf=True)
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

# 5. Train & Evaluate Multinomial Naive Bayes
nb = MultinomialNB(alpha=0.2)
nb.fit(X_train_vec, y_train)
y_pred = nb.predict(X_test_vec)

print("Accuracy :", round(accuracy_score(y_test, y_pred)*100, 2), "%")
print("Precision:", round(precision_score(y_test, y_pred)*100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))